# 05 — SCIN Dataset: Gaussian Noise Augmentation Experiment

Downloads the SCIN dataset from its public GCS bucket, prepares it for
the skin tone classification pipeline, and trains with Gaussian noise augmentation.

**References:**
- SCIN dataset: https://github.com/google-research-datasets/scin
- Augmentation study: https://pmc.ncbi.nlm.nih.gov/articles/PMC5977656/
- TorchIO noise: https://www.sciencedirect.com/science/article/pii/S0169260721003102

In [ ]:
# Cell 1: Setup
import os, subprocess, sys

if not os.path.exists("/content/NST_Class"):
    subprocess.run(["git", "clone", "https://github.com/AayushBaniya2006/NST_Class.git"], cwd="/content")
os.chdir("/content/NST_Class")

!pip install -q -r requirements.txt
sys.path.insert(0, "/content/NST_Class")

print("Setup complete!")

In [ ]:
# Cell 2: Download SCIN dataset + prepare splits + upload to your bucket
!python scripts/prepare_scin.py --upload-bucket skin-tone-project

In [ ]:
# Cell 3: Verify — count images and check splits
import glob
import pandas as pd

# Count raw images downloaded
image_files = glob.glob("data/scin/images/**/*", recursive=True)
image_files = [f for f in image_files if os.path.isfile(f)]
print(f"Total images downloaded: {len(image_files)}")

# Load and verify splits
train_df = pd.read_csv("data/scin_cleaned/train.csv")
val_df = pd.read_csv("data/scin_cleaned/val.csv")
test_df = pd.read_csv("data/scin_cleaned/test.csv")

print(f"\nSplit sizes:")
print(f"  Train: {len(train_df)}")
print(f"  Val:   {len(val_df)}")
print(f"  Test:  {len(test_df)}")
print(f"  Total: {len(train_df) + len(val_df) + len(test_df)}")

# Check class distribution per split
print(f"\nClass distribution (train):")
for label in sorted(train_df["skin_tone_label"].unique()):
    count = (train_df["skin_tone_label"] == label).sum()
    print(f"  FST{label+1} (label {label}): {count} ({100*count/len(train_df):.1f}%)")

# Verify images actually exist on disk for a sample of rows
from src.data.prepare import _find_image_path

missing = 0
sample = train_df.head(50)
for _, row in sample.iterrows():
    path = _find_image_path("data/scin/images", row["hasher"])
    if path is None:
        missing += 1

print(f"\nImage check (first 50 train rows): {50 - missing}/50 found")
if missing > 0:
    print(f"  WARNING: {missing} images missing!")
else:
    print("  All images found on disk.")

In [ ]:
# Cell 4: Sanity check — load one batch and verify transforms work
import torch
from torch.utils.data import DataLoader
from src.data.dataset import FitzpatrickDataset
from src.data.transforms import get_train_transforms, GaussianNoise

transform = get_train_transforms(224, augmentation="noise")
print(f"Transform pipeline:\n{transform}\n")

dataset = FitzpatrickDataset(train_df, "data/scin/images", transform=transform)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

images, labels = next(iter(loader))
print(f"Batch shape:  {images.shape}")   # expect [8, 3, 224, 224]
print(f"Labels:       {labels.tolist()}")
print(f"Pixel range:  [{images.min():.3f}, {images.max():.3f}]")
print(f"\nSanity check PASSED — pipeline works!")

In [ ]:
# Cell 5: Train with Gaussian noise augmentation
!python scripts/train.py --augmentation noise \
    --data-dir data/scin_cleaned \
    --image-dir data/scin/images

In [ ]:
# Cell 6: Verify training output
import glob

checkpoints = glob.glob("checkpoints/*.pt")
print(f"Checkpoints saved: {len(checkpoints)}")
for cp in checkpoints:
    size_mb = os.path.getsize(cp) / (1024 * 1024)
    print(f"  {cp} ({size_mb:.1f} MB)")

if not checkpoints:
    print("ERROR: No checkpoints found! Training may have failed.")

In [ ]:
# Cell 7: Upload checkpoint to your bucket
!gsutil -m cp -r checkpoints/ gs://skin-tone-project/checkpoints/noise/

In [ ]:
# Cell 8: Save to Google Drive (backup)
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/NST_scin_checkpoints
!cp -r checkpoints/* /content/drive/MyDrive/NST_scin_checkpoints/
print("Checkpoint backed up to Google Drive!")